In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import ipywidgets as widgets
from IPython.display import display, clear_output

folder_names = ["highway", "pedestrian", "office"]

def load_folder_name(folder_name: str) -> tuple[list[np.ndarray], list[np.ndarray], np.ndarray]:
    """Return grayscale frames, ground truth masks, and ROI mask."""
    roi = cv2.imread(f"{folder_name}/ROI.jpg")
    roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    roi = roi > 0

    with open(f"{folder_name}/temporalROI.txt", "r") as f:
        line = f.readline()
        a, b = map(int, line.split(" "))

    imgs = [None] * (b - a + 1)
    gts = [None] * (b - a + 1)
    for i in range(a, b + 1):
        path = str(i).zfill(6)
        gt = folder_name + "/groundtruth/" + "gt" + path + ".png"
        img = folder_name + "/input/" + "in" + path + ".jpg"
        gt_img = cv2.imread(gt)
        gt_img = cv2.cvtColor(gt_img, cv2.COLOR_BGR2GRAY) if gt_img is not None else None
        img_data = cv2.imread(img)
        img_data = cv2.cvtColor(img_data, cv2.COLOR_BGR2GRAY) if img_data is not None else None

        if gt_img is not None and img_data is not None:
            gts[i - a] = gt_img
            imgs[i - a] = img_data
        else:
            print(f"i={i}, Błąd przy wczytywaniu")
    print(len(imgs))
    print(len(gts))
    print(b-a+1)
    return imgs, gts, roi,0,b-a


In [ ]:
folder_name = "highway"
imgs, gts, roi,a,b= load_folder_name(folder_name)

In [ ]:
def get_labels(diffs: list[np.ndarray]):
    labels_ = []
    for idx in range(0,len(diffs)):
        retval, labels, stats, centroids = cv2.connectedComponentsWithStats(diffs[idx])
        labels = cv2.normalize(labels, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        labels_.append(labels)

    return labels_
def play_frames(
    imgs: list[np.ndarray],
    gts: list[np.ndarray],
    results: list[np.ndarray],
    names: list[str],
) -> None:
    """Simple OpenCV player. Close the window (or press q / Esc) to stop."""
    max_idx = min(len(imgs) - 1, len(gts) - 1, len(results[0]) - 1 if results else -1)
    if max_idx < 0:
        raise ValueError("Empty input arrays.")
    interval_ms = 100
    start = 50
    step = 2
    i = start
    window_name = "Frames"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    print("getting labels")
    labels = get_labels(results[0])
    
    while True:
        frame_list = []
        if i > max_idx:
            i = start
        # Convert input and GT to BGR
        img_bgr = cv2.cvtColor(imgs[i], cv2.COLOR_GRAY2BGR)
        gt_bgr = cv2.cvtColor(gts[i].astype(np.uint8), cv2.COLOR_GRAY2BGR)

        frame_list.append(img_bgr)
        frame_list.append(gt_bgr)

        # Add each result with name label
        for res, name in zip(results, names):
            res_img = res[i].astype(np.uint8)
            # Convert grayscale to BGR for colored text
            if len(res_img.shape) == 2:
                res_img = cv2.cvtColor(res_img, cv2.COLOR_GRAY2BGR)
            cv2.putText(res_img, name, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            frame_list.append(res_img)

        # Add labels
        labels_img = labels[i]
        if len(labels_img.shape) == 2:
            labels_img = cv2.cvtColor(labels_img, cv2.COLOR_GRAY2BGR)
        frame_list.append(labels_img)

        frame = cv2.hconcat(frame_list)
        cv2.imshow(window_name, frame)

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

        key = cv2.waitKey(interval_ms) & 0xFF
        if key == ord("q") or key == 27:
            break

        i += step
      

    cv2.destroyWindow(window_name)

In [ ]:
def morph(imgs: list[np.ndarray],roi:np.ndarray) -> list[np.ndarray]:
    """Compute motion masks from consecutive frames."""
    morphed = np.zeros_like(imgs)
    for i in range(0, len(imgs)):
        im = imgs[i]
        _, thresh = cv2.threshold(im, 20, 255, cv2.THRESH_BINARY)
        thresh = (thresh & roi * 255).astype("uint8")
        im_org = thresh
        im = cv2.medianBlur(im_org, 3)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        im = cv2.morphologyEx(im, cv2.MORPH_ERODE, kernel)
       
        while True:
            im2 = cv2.morphologyEx(im, cv2.MORPH_DILATE, kernel)
            im3 = im2 & im_org
            if (im3 == im).all():
                break
            im = im3

        im = cv2.morphologyEx(im, cv2.MORPH_CLOSE, kernel)
     
     
        morphed[i] = im
    return morphed
def morph_one(img: np.ndarray,roi:np.ndarray) -> np.ndarray:
    """Compute motion masks from consecutive frames."""
    im = img
    _, thresh = cv2.threshold(im, 2, 255, cv2.THRESH_BINARY)
    thresh = thresh.astype(bool)
    thresh = (thresh > 0) & roi
    thresh = (thresh.astype(np.uint8) * 255)
    im_org = thresh
    im = cv2.medianBlur(im_org, 3)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    im = cv2.morphologyEx(im, cv2.MORPH_ERODE, kernel)
    while True:
        im2 = cv2.morphologyEx(im, cv2.MORPH_DILATE, kernel)
        im3 = im2 & im_org
        if (im3 == im).all():
            break
        im = im3

    im = cv2.morphologyEx(im, cv2.MORPH_CLOSE, kernel)
     
     
     
    return im


In [ ]:
N=60
imgs = np.array(imgs).astype(np.uint8)
idx = 0
BUF = np.zeros((N,imgs[0].shape[0],imgs[0].shape[1]),np.uint8)
medians_fg = []
means_fg = []
means_sigma_fg = []
medians_sigma_fg = []
alpha = 0.01
B_means = np.zeros(imgs.shape,dtype=np.float32)
B_medians = np.zeros(imgs.shape)
B_medians_fg = np.zeros(imgs.shape,dtype=np.float32)
B_means_fg = np.zeros(imgs.shape,dtype=np.float32)
conservative = np.zeros(imgs.shape)
B_medians_2 = np.zeros(imgs.shape)
B_medians_2_fg = np.zeros(imgs.shape)
for i in range(len(imgs)):

    B_means[i] = alpha*imgs[i]+(1-alpha)*(B_means[i-1] if i>0 else 0)
    prev = B_medians[i-1] if i>0 else imgs[i]
    curr = imgs[i]
    B_medians[i]=prev + (curr>prev) - (curr<prev)
    mean = np.mean(BUF,axis = 0, dtype=np.float64)
    median = np.median(BUF,axis=0)
    median_fg = np.abs(imgs[i] - median)
    mean_fg = np.abs(imgs[i] - mean)
    B_medians_fg[i] = np.abs(imgs[i] - B_medians[i])
    B_means_fg[i] = np.abs(imgs[i] - B_means[i])
    # _, median_fg = cv2.threshold(median_fg, 30, 255, cv2.THRESH_BINARY)
    # median_fg = (median_fg & roi * 255).astype("uint8")
    #_, mean_fg = cv2.threshold(mean_fg, 30, 255, cv2.THRESH_BINARY)
    # mean_fg = (mean_fg & roi * 255).astype("uint8")
    means_fg.append(mean_fg)
    medians_fg.append(median_fg)


    prev2 = B_medians_2[i-1].astype("int16") if i>0 else imgs[i].astype("int16")
    curr2 = imgs[i].astype("int16")
    #prev2 == 0 is true only only when it was background in the previous one
    diff = np.abs(curr2-prev2).astype("int8")
    mask = diff < 20
    B_medians_2[i] = prev2 + (curr2>prev2)*(mask) - (curr2<prev2) * (mask)
    B_medians_2_fg[i] = np.abs(imgs[i].astype("int16") - B_medians_2[i].astype("int16"))
    




    
    # T = 25.0  # prog klasyfikacji BG/FG (dobierz eksperymentalnie)

    # if i == 0:
    #     B_medians_2[i] = imgs[i].astype(np.float32)
    # else:
    #     prev2 = B_medians_2[i - 1]
    #     curr2 = imgs[i].astype(np.float32)

    #     # klasyfikacja: male odchylenie od modelu = tlo
    #     diff = np.abs(curr2 - prev2)
    #     bg_mask = diff < T

    #     # opcjonalnie ograniczenie do ROI:
    #     # bg_mask = bg_mask & roi

    #     # mediana przyrostowa: +1 / -1 / 0, ale tylko dla tla
    #     step = np.sign(curr2 - prev2)
    #     B_medians_2[i] = prev2 + step * bg_mask.astype(np.float32)

    # B_medians_2_fg[i] = np.abs(imgs[i].astype(np.float32) - B_medians_2[i])





    BUF[idx] = imgs[i]
    idx = (idx+1)%N
  




In [ ]:
def scoring(results,gts,a,b):
    gts = np.array(gts)
    results = np.array(results)
    p = 0
    r = 0
    f1 = 0
    gts = (gts>0)
    results=(results>0)
    for i in range(a, b):
        gt = (gts[i] > 0)
        pred = (results[i] > 0)

        tp = np.sum(gt & pred)
        fp = np.sum(~gt & pred)
        fn = np.sum(gt & ~pred)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        F1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        p += precision
        r += recall
        f1 += F1
    count = b - a
    p /= count
    r /= count
    f1 /= count
    print(f"precision {p}, recall {r}, f1 {f1}",end="")


In [ ]:
fgbg = cv2.createBackgroundSubtractorMOG2(history = 60, varThreshold = 16, detectShadows = False)
gaussian = np.zeros(imgs.shape).astype("uint8")
for i in range(len(imgs)):
    gaussian[i] = fgbg.apply(imgs[i]).astype("uint8")
knn = np.zeros(imgs.shape).astype("uint8")
knn_sub = cv2.createBackgroundSubtractorKNN(history=60,detectShadows=False)
for i in range(len(imgs)):
    knn[i] = knn_sub.apply(imgs[i]).astype("uint8")



In [ ]:
medians_fg = np.array(medians_fg).astype(np.uint8)
means_fg = np.array(means_fg).astype(np.uint8)
B_means_fg  = np.array(B_means_fg).astype(np.uint8)
B_medians_fg = np.array(B_medians_fg).astype(np.uint8)
B_medians_2_fg = np.array(B_medians_2_fg).astype(np.uint8)

fgs = [medians_fg,means_fg,B_means_fg,B_medians_fg,B_medians_2_fg,gaussian,knn]
fgs_names = ["medians","means","B_means","B_medians","conservative median","gaussian","knn"]
fgs_morphed = []
for fg in fgs:
    fgs_morphed.append(morph(fg,roi))

In [ ]:
play_frames(imgs=imgs,gts=gts,results=fgs_morphed,names = fgs_names)

In [ ]:
plt.imshow(fgs_morphed[4][340],cmap="gray")

In [ ]:
plt.imshow(fgs_morphed[3][340],cmap="gray")

In [ ]:
for fg_name, fg in zip(fgs_names, fgs_morphed):
    print(f"scoring of {fg_name}: ",end="")
    scoring(fg,gts,a,b)
    print("\n")